# LeetCode 991 — Broken Calculator bằng Python

Link đề: https://leetcode.com/problems/broken-calculator/ · Độ khó: **Medium** · Chủ đề: Math, Greedy

Skeleton LeetCode:

```python
class Solution:
    def brokenCalc(self, startValue: int, target: int) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (số phép tính tối thiểu). Điểm cốt lõi: nghĩ **ngược** từ `target` về `startValue` — phép ngược của "nhân đôi" là "chia đôi", phép ngược của "trừ 1" là "cộng 1"; khi `target` lẻ **bắt buộc** cộng 1, khi chẵn thì chia đôi. Oracle độc lập: **BFS thuận** trên đồ thị các giá trị.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ `startValue` chỉ được **nhân đôi** hoặc **trừ 1**; tìm số phép tối thiểu để đạt `target`.
2. Nắm công cụ: BFS trên đồ thị số, tham lam **ngược**, đệ quy, đếm phép bằng **số lần nhân đôi** `k`, phân tích nhị phân.
3. Đi từ **BFS thuận** → **BFS ngược có chặn** → **tham lam ngược (vòng lặp)** → **tham lam ngược (đệ quy)** → **công thức theo số lần nhân đôi `k`**.
4. Hiểu mấu chốt: ngược lại, `target` **lẻ** thì chỉ có thể do "trừ 1" từ `target + 1` ⇒ cộng 1; `target` **chẵn** thì chia đôi **luôn tốt hơn** cộng 1 (chứng minh bằng lập luận đổi thứ tự phép); khi `target ≤ startValue` thì chỉ còn `startValue − target` phép trừ.
5. Tránh lỗi kinh điển: tham lam **thuận** (nhân đôi tới khi vượt rồi trừ), trừ 1 thay vì cộng 1 khi `target` lẻ, quên cộng `startValue − target` ở cuối, vòng `while target != startValue` lặp vô hạn khi `target < startValue`, chia nguyên `//` ở số lẻ.

## 2. Đề bài & ví dụ

Máy tính hỏng đang hiển thị số nguyên `startValue`. Mỗi phép tính có thể:

- **nhân** số đang hiển thị với `2`, hoặc
- **trừ** `1` khỏi số đang hiển thị.

Cho `startValue` và `target`, trả về số phép tính **tối thiểu** để hiển thị `target`.

**Ví dụ 1:** `startValue = 2, target = 3` → `2` (`2 → 4 → 3`).

**Ví dụ 2:** `startValue = 5, target = 8` → `2` (`5 → 4 → 8`).

**Ví dụ 3:** `startValue = 3, target = 10` → `3` (`3 → 6 → 5 → 10`).

**Ràng buộc:** `1 <= startValue, target <= 10^9`.

In [1]:
import itertools
import random
import time
from collections import deque
from typing import List

examples = [((2, 3), 2), ((5, 8), 2), ((3, 10), 3), ((1024, 1), 1023)]
for (s, t), want in examples:
    print(f"startValue = {s}, target = {t} -> {want}")

startValue = 2, target = 3 -> 2
startValue = 5, target = 8 -> 2
startValue = 3, target = 10 -> 3
startValue = 1024, target = 1 -> 1023


## 3. Kiến thức nền

### 3.1 Đồ thị các giá trị

Mỗi số nguyên dương là một **đỉnh**; từ `v` có cạnh tới `2v` (nhân đôi) và `v − 1` (trừ 1, nếu `v − 1 ≥ 1`; giá trị `≤ 0` không bao giờ đưa về `target ≥ 1` vì nhân đôi `≤ 0` vẫn `≤ 0` và trừ làm nhỏ thêm). Số phép tối thiểu = **khoảng cách ngắn nhất** từ `startValue` đến `target` (BFS).

### 3.2 Phép ngược

Ngược lại của "nhân đôi" là "chia đôi" (chỉ với số **chẵn**); ngược lại của "trừ 1" là "cộng 1". Đi ngược từ `target` về `startValue` cho cùng số phép.

### 3.3 Số lẻ và số chẵn khi đi ngược

Số **lẻ** `t` không thể là kết quả của nhân đôi (nhân đôi cho số chẵn) ⇒ nó chỉ có thể do "trừ 1" từ `t + 1` ⇒ phép ngược duy nhất là `t → t + 1`. Số **chẵn** `t` có thể do nhân đôi từ `t/2` hoặc trừ 1 từ `t + 1`.

In [2]:
def parity_demo(t):
    return ("lẻ: chỉ có thể ngược bằng +1 -> " + str(t + 1)) if t % 2 else ("chẵn: ngược bằng /2 -> " + str(t // 2) + " (hoặc +1 -> " + str(t + 1) + ")")


for t in (3, 8, 10, 7):
    print(t, parity_demo(t))

3 lẻ: chỉ có thể ngược bằng +1 -> 4
8 chẵn: ngược bằng /2 -> 4 (hoặc +1 -> 9)
10 chẵn: ngược bằng /2 -> 5 (hoặc +1 -> 11)
7 lẻ: chỉ có thể ngược bằng +1 -> 8


## 4. Ý tưởng nền tảng / trực giác

`startValue = 3, target = 10`. Đi **ngược** từ `10`:

- `10` chẵn ⇒ chia đôi: `5` (1 phép);
- `5` lẻ ⇒ cộng 1: `6` (2 phép);
- `6` chẵn ⇒ chia đôi: `3` (3 phép) = `startValue`.

Tổng `3` phép, đúng ví dụ 3 (`3 → 6 → 5 → 10`). Khi `target` đã **không lớn hơn** `startValue` thì chỉ còn trừ 1 `startValue − target` lần (không thể tăng bằng cách khác có lợi).

Nghĩ **thuận** (nhân đôi tới khi vượt `target` rồi trừ dần) thường sai: ví dụ `(5, 8)`: nhân đôi `5 → 10`, trừ 2 lần ⇒ 3 phép; trong khi `5 → 4 → 8` chỉ 2 phép (trừ **trước**, nhân đôi sau).

In [3]:
def trace_reverse_print(s, t):
    ops, steps = 0, [t]
    while t > s:
        t = t + 1 if t % 2 else t // 2
        ops += 1
        steps.append(t)
    print(f"s={s}: đường ngược {steps} + {s - t} lần cộng... (trừ {s - t})", "=> tổng", ops + (s - t))


trace_reverse_print(3, 10)
trace_reverse_print(5, 8)

s=3: đường ngược [10, 5, 6, 3] + 0 lần cộng... (trừ 0) => tổng 3
s=5: đường ngược [8, 4] + 1 lần cộng... (trừ 1) => tổng 2


## 5. Brute force — BFS thuận

**Ý tưởng:** BFS từ `startValue`; từ `v` đi tới `2v` và `v − 1`. Giới hạn các giá trị trong `[1, U]` với `U` đủ lớn (`U = 4·max(startValue, target) + 4`) để BFS hữu hạn; ta kiểm chứng ở mục 12 rằng nâng `U` không đổi đáp án. Đây là **oracle độc lập** (không dùng bất kỳ lập luận tham lam nào).

In [4]:
def broken_bfs(start, target, bound=None):
    U = bound if bound is not None else 4 * max(start, target) + 4
    dist = {start: 0}
    queue = deque([start])
    while queue:
        v = queue.popleft()
        if v == target:
            return dist[v]
        for w in (2 * v, v - 1):
            if 1 <= w <= U and w not in dist:
                dist[w] = dist[v] + 1
                queue.append(w)
    return -1


for (s, t), want in examples:
    assert broken_bfs(s, t) == want
print([broken_bfs(s, t) for (s, t), _ in examples])

[2, 2, 3, 1023]


In [5]:
# Code nộp LeetCode cho Brute force (BFS giới hạn; chỉ dùng cho giá trị nhỏ)
from collections import deque


class SolutionBFS:
    def brokenCalc(self, startValue: int, target: int) -> int:
        U = 4 * max(startValue, target) + 4
        dist = {startValue: 0}
        queue = deque([startValue])
        while queue:
            v = queue.popleft()
            if v == target:
                return dist[v]
            for w in (2 * v, v - 1):
                if 1 <= w <= U and w not in dist:
                    dist[w] = dist[v] + 1
                    queue.append(w)
        return -1

**Độ phức tạp:** `O(U)` thời gian và bộ nhớ — với `10^9` bất khả thi.

**Nút thắt:** số đỉnh phình ra (cả `v-1` và `2v`). Nhưng đi **ngược** từ `target` thì mỗi đỉnh gần như chỉ có **một** lựa chọn hợp lý.

## 6. Bước đột phá

Ngược từ `target`: nếu `target` lẻ ⇒ `+1` (bắt buộc); nếu chẵn ⇒ `/2` (luôn tối ưu). Lặp tới khi `target ≤ startValue`, rồi cộng `startValue − target` phép trừ. Mỗi hai bước giá trị giảm ít nhất một nửa ⇒ `O(log target)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** tham lam ngược bằng vòng lặp (đáp án chính), **Cách 3** tham lam ngược bằng đệ quy, **Cách 4** BFS ngược có chặn, **Cách 5** công thức theo số lần nhân đôi `k`.

### Cách 2 — Tham lam ngược (vòng lặp)

**Giải thích từng đoạn:**

1. `ops = 0`.
2. `while target > startValue`: nếu `target` lẻ ⇒ `target += 1`, ngược lại `target //= 2`; `ops += 1`.
3. Khi `target ≤ startValue`: còn `startValue − target` phép trừ ⇒ `return ops + startValue − target`.

In [6]:
def broken_greedy(start, target):
    ops = 0
    while target > start:
        if target % 2:
            target += 1                      # lẻ: chỉ có thể do "trừ 1" từ target + 1
        else:
            target //= 2                     # chẵn: chia đôi (ngược của nhân đôi)
        ops += 1
    return ops + start - target             # target <= start: còn start - target phép trừ


for (s, t), want in examples:
    assert broken_greedy(s, t) == want
print([broken_greedy(s, t) for (s, t), _ in examples])

[2, 2, 3, 1023]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionGreedy:
    def brokenCalc(self, startValue: int, target: int) -> int:
        ops = 0
        while target > startValue:
            target = target + 1 if target % 2 else target // 2
            ops += 1
        return ops + startValue - target

**Độ phức tạp:** `O(log target)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Tham lam ngược (đệ quy)

**Ý tưởng:** `f(s, t) = s − t` nếu `t ≤ s`; `1 + f(s, t/2)` nếu `t` chẵn; `1 + f(s, t+1)` nếu `t` lẻ. Độ sâu `O(log t)` (`≤ 62`).

In [8]:
def broken_recursive(s, t):
    if t <= s:
        return s - t
    if t % 2 == 0:
        return 1 + broken_recursive(s, t // 2)
    return 1 + broken_recursive(s, t + 1)


for (s, t), want in examples:
    assert broken_recursive(s, t) == want
print([broken_recursive(s, t) for (s, t), _ in examples])

[2, 2, 3, 1023]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionRecursive:
    def brokenCalc(self, startValue: int, target: int) -> int:
        if target <= startValue:
            return startValue - target
        if target % 2 == 0:
            return 1 + self.brokenCalc(startValue, target // 2)
        return 1 + self.brokenCalc(startValue, target + 1)

**Độ phức tạp:** `O(log target)` thời gian và ngăn xếp.

### Cách 4 — BFS ngược có chặn

**Ý tưởng:** BFS từ `target` với hai phép ngược: `x → x + 1` và (nếu `x` chẵn) `x → x / 2`; dừng khi gặp `startValue`. Giới hạn `x ≤ 2·max(startValue, target)` để hữu hạn. Cách này **không** dùng lập luận "chẵn thì chia đôi", nhưng vẫn dựa vào phép ngược; chạy được cho giá trị trung bình nhờ chặn.

In [10]:
def broken_reverse_bfs(start, target):
    U = 2 * max(start, target) + 2
    dist = {target: 0}
    queue = deque([target])
    while queue:
        x = queue.popleft()
        if x == start:
            return dist[x]
        nxt = [x + 1]
        if x % 2 == 0:
            nxt.append(x // 2)
        for y in nxt:
            if 1 <= y <= U and y not in dist:
                dist[y] = dist[x] + 1
                queue.append(y)
    return -1


for (s, t), want in examples[:3]:
    assert broken_reverse_bfs(s, t) == want
print([broken_reverse_bfs(s, t) for (s, t), _ in examples[:3]])

[2, 2, 3]


In [11]:
# Code nộp LeetCode cho Cách 4 (BFS ngược có chặn; chỉ dùng cho giá trị nhỏ/trung bình)
from collections import deque


class SolutionReverseBFS:
    def brokenCalc(self, startValue: int, target: int) -> int:
        U = 2 * max(startValue, target) + 2
        dist = {target: 0}
        queue = deque([target])
        while queue:
            x = queue.popleft()
            if x == startValue:
                return dist[x]
            for y in ((x + 1, x // 2) if x % 2 == 0 else (x + 1,)):
                if 1 <= y <= U and y not in dist:
                    dist[y] = dist[x] + 1
                    queue.append(y)
        return -1

**Độ phức tạp:** `O(max(start, target))` trường hợp xấu; nhanh hơn BFS thuận vì mỗi đỉnh chỉ có ≤ 2 cạnh ngược và phần lớn giá trị lẻ chỉ có một.

### Cách 5 — Công thức theo số lần nhân đôi `k`

**Ý tưởng:** giả sử dùng đúng `k` lần nhân đôi. Gọi `d_i ≥ 0` là số lần trừ 1 thực hiện sau `i` lần nhân đôi (`i = 0..k`). Giá trị cuối `= s·2^k − Σ d_i · 2^{k−i}`. Đặt `D = s·2^k − target ≥ 0`. Cần biểu diễn `D = Σ d_i · 2^{k−i}` với **tổng `d_i` nhỏ nhất**: đây là bài toán "đổi tiền" với mệnh giá `1, 2, 4, …, 2^k` (mệnh giá lớn nhất `2^k` có số lượng tuỳ ý): tối ưu là lấy `D // 2^k` đồng `2^k`, phần dư `D mod 2^k` biểu diễn nhị phân ⇒ `popcount` đồng. Vậy số phép với `k` lần nhân đôi là `k + D // 2^k + popcount(D mod 2^k)`; lấy `min` theo `k`.

In [12]:
def broken_by_k(start, target):
    best = None
    k = 0
    while True:
        D = start * (1 << k) - target
        if D >= 0:
            cost = k + D // (1 << k) + bin(D % (1 << k)).count("1")
            best = cost if best is None else min(best, cost)
        if start * (1 << k) >= target and k > 0 and D >= (1 << k) * start:
            break                                           # D lớn hơn bao nhiêu thì k lớn hơn cũng không tốt hơn
        k += 1
        if k > 64:
            break
    return best


for (s, t), want in examples:
    assert broken_by_k(s, t) == want
print([broken_by_k(s, t) for (s, t), _ in examples])

[2, 2, 3, 1023]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionByK:
    def brokenCalc(self, startValue: int, target: int) -> int:
        best = float("inf")
        for k in range(0, 40):
            D = startValue * (1 << k) - target
            if D >= 0:
                best = min(best, k + D // (1 << k) + bin(D % (1 << k)).count("1"))
        return best

**Độ phức tạp:** `O(log² )` — `k` chạy tới `~31` (vì `s·2^k ≥ target` rồi `D` tăng gấp đôi mỗi bước, ta chỉ cần `k ≤ log₂ target + 1`), mỗi bước `O(popcount)`.

## 8. TRACE từng bước

**Trace Cách 2** với `startValue = 3`, `target = 10`:

| bước | `target` trước | chẵn/lẻ | phép ngược | `target` sau | `ops` |
|---|---|---|---|---|---|
| 1 | 10 | chẵn | `/ 2` | 5 | 1 |
| 2 | 5 | lẻ | `+ 1` | 6 | 2 |
| 3 | 6 | chẵn | `/ 2` | 3 | 3 |
| kết | 3 | `3 ≤ 3` | còn `3 − 3 = 0` phép trừ | — | **3** |

Hàm trace in từng bước (kèm đường đi thuận tương ứng):

In [14]:
def trace_greedy(s, t):
    print(f"startValue = {s}, target = {t}")
    ops, path, cur = 0, [], t
    while cur > s:
        if cur % 2:
            print(f"  {cur} lẻ  -> +1 -> {cur + 1}")
            path.append("-1")
            cur += 1
        else:
            print(f"  {cur} chẵn -> /2 -> {cur // 2}")
            path.append("*2")
            cur //= 2
        ops += 1
    extra = s - cur
    print(f"  {cur} <= {s}: cần thêm {extra} phép trừ 1")
    forward = ["-1"] * extra + path[::-1]                 # đi thuận: trừ extra lần, rồi các phép (đảo ngược danh sách)
    v = s
    seq = [v]
    for op in forward:
        v = v * 2 if op == "*2" else v - 1
        seq.append(v)
    print("  đường đi thuận:", " -> ".join(map(str, seq)), f"({len(forward)} phép)")
    assert v == t and len(forward) == ops + extra
    return ops + extra


assert trace_greedy(3, 10) == 3
assert trace_greedy(5, 8) == 2

startValue = 3, target = 10
  10 chẵn -> /2 -> 5
  5 lẻ  -> +1 -> 6
  6 chẵn -> /2 -> 3
  3 <= 3: cần thêm 0 phép trừ 1
  đường đi thuận: 3 -> 6 -> 5 -> 10 (3 phép)
startValue = 5, target = 8
  8 chẵn -> /2 -> 4
  4 <= 5: cần thêm 1 phép trừ 1
  đường đi thuận: 5 -> 4 -> 8 (2 phép)


**Trace trường hợp `target ≤ startValue`:** `(1024, 1)`: vòng lặp không chạy; đáp án `1024 − 1 = 1023` (chỉ có thể trừ 1; nhân đôi chỉ làm xa).

In [15]:
assert broken_greedy(1024, 1) == 1023 and broken_bfs(1024, 1) == 1023 and broken_greedy(7, 7) == 0
print(broken_greedy(1024, 1), broken_greedy(7, 7))

1023 0


## 9. Code nộp LeetCode

Chọn **Cách 2** (tham lam ngược, vòng lặp). Giữ nguyên `class Solution`, `brokenCalc(self, startValue, target)`:

In [16]:
class Solution:
    def brokenCalc(self, startValue: int, target: int) -> int:
        ops = 0
        while target > startValue:                 # đi ngược từ target về startValue
            if target % 2:
                target += 1                        # lẻ: phép ngược duy nhất là +1
            else:
                target //= 2                       # chẵn: chia đôi
            ops += 1
        return ops + startValue - target           # target <= startValue: chỉ còn các phép trừ 1


print([Solution().brokenCalc(s, t) for (s, t), _ in examples])
assert all(Solution().brokenCalc(s, t) == w for (s, t), w in examples)

[2, 2, 3, 1023]


## 10. Vì sao đúng

**(a) Phép ngược và đồ thị.** Số phép tối thiểu từ `s` tới `t` (theo `×2`, `−1`) bằng số phép tối thiểu từ `t` về `s` theo các phép ngược `÷2` (chỉ khi chẵn) và `+1`. Giá trị `≤ 0` không hữu ích (nhân đôi `≤ 0` không tăng, trừ làm giảm) nên ta chỉ xét số dương.

**(b) `target` lẻ ⇒ `+1` bắt buộc.** Nhân đôi cho số chẵn; do đó số lẻ `t` chỉ có thể là kết quả của `−1` từ `t + 1`. Ngược lại: `t → t + 1`.

**(c) `target` chẵn: chia đôi tối ưu.** Xét `t` chẵn, `t > s`, và một đường ngược tối ưu từ `t` về `s`. Vì `s < t` và chỉ phép `÷2` làm giá trị giảm nhiều, đường đó phải có ít nhất một phép `÷2`. Gọi phép `÷2` **đầu tiên** xảy ra tại giá trị `t + j` (trước đó chỉ có `j` phép `+1`; vì `t + j` phải chẵn nên `j` chẵn). Đường này tốn `j + 1` phép để tới `(t + j)/2 = t/2 + j/2`. Đường thay thế: `÷2` ngay từ `t` rồi `+1` đúng `j/2` lần cũng tới `t/2 + j/2` nhưng chỉ tốn `1 + j/2` phép. Mà `1 + j/2 ≤ j + 1`, dấu bằng chỉ khi `j = 0`. Vậy tồn tại đường tối ưu bắt đầu bằng `÷2` (nếu `j ≥ 2` đường thay thế còn **ngắn hơn**, mâu thuẫn với tính tối ưu; nên trong đường tối ưu `j = 0`).

**(d) `target ≤ startValue` ⇒ đáp án `s − t`.** Với `k` lần nhân đôi, theo (f) số phép là `k + D // 2^k + popcount(D mod 2^k) ≥ k + D / 2^k = k + s − t / 2^k ≥ s − t` (vì `k ≥ 0` và `t − t/2^k ≥ 0`), dấu bằng ở `k = 0`. Vậy không thể rẻ hơn `s − t`.

**(e) Số bước.** Mỗi hai bước (một `+1` rồi `÷2`) giá trị giảm ít nhất một nửa ⇒ `O(log t)` bước.

**(f) Cách 5.** Với `k` lần nhân đôi cố định, tổng số phép là `k +` số lần trừ. Các lần trừ ở giai đoạn `i` giảm giá trị cuối đi `2^{k−i}`; cần `Σ d_i 2^{k−i} = D = s·2^k − t`. Đổi `D` thành tổng các luỹ thừa `2^{k−i}` (`0 ≤ i ≤ k`) ít số hạng nhất: lấy mệnh giá lớn nhất `2^k` nhiều nhất có thể (`D // 2^k` lần), phần dư `< 2^k` biểu diễn nhị phân ít số hạng nhất (`popcount`) — đổi tiền với mệnh giá luỹ thừa 2 là tối ưu theo tham lam. Điều kiện thực hiện được (giá trị trung gian dương) thoả vì `d_0 ≤ D // 2^k ≤ s − 1` (do `t ≥ 1`).

**(g) Kiểm chứng bằng máy.** Mọi cách (trừ BFS ngược có chặn giới hạn giá trị) khớp oracle BFS thuận trên **mọi** cặp `1 ≤ startValue, target ≤ 300` (`90 000` cặp), BFS ngược khớp ở miền nhỏ, và các cách `O(log)` khớp nhau tới `10^9`; mỗi đáp án lớn được **dựng thành đường đi cụ thể** rồi phát lại để kiểm số phép và giá trị cuối (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS thuận | `O(U)` | `O(U)` | oracle giá trị nhỏ |
| 2. Tham lam ngược (vòng lặp) | `O(log t)` | `O(1)` | **đáp án chính** |
| 3. Tham lam ngược (đệ quy) | `O(log t)` | `O(log t)` | cùng ý, dạng đệ quy |
| 4. BFS ngược có chặn | `O(max(s,t))` | `O(max(s,t))` | kiểm chứng không cần chứng minh tham lam |
| 5. Công thức theo `k` | `O(log² t)` | `O(1)` | cách nhìn khác (đổi tiền) |

Đo thực tế:

In [17]:
def bench(fn, args, repeat=2000):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


for name, fn in [("greedy", broken_greedy), ("recursive", broken_recursive), ("by_k", broken_by_k)]:
    print(f"{name:10s}: {bench(fn, (1, 10 ** 9)) * 1e6:7.2f} us (s = 1, t = 10^9) -> {fn(1, 10 ** 9)}")
print(f"{'bfs':10s}: {bench(broken_bfs, (3, 20000), repeat=3) * 1e3:7.2f} ms (s = 3, t = 20000)")
print(f"{'reverse':10s}: {bench(broken_reverse_bfs, (3, 20000), repeat=3) * 1e3:7.2f} ms (s = 3, t = 20000)")

greedy    :    2.33 us (s = 1, t = 10^9) -> 39
recursive :    3.06 us (s = 1, t = 10^9) -> 39


by_k      :   25.21 us (s = 1, t = 10^9) -> 39


bfs       :    4.07 ms (s = 3, t = 20000)
reverse   :    0.04 ms (s = 3, t = 20000)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi cặp `1 ≤ s, t ≤ 300` với oracle BFS thuận, (E) tính chất / ngẫu nhiên (seed cố định, tới `10^9`), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "greedy": broken_greedy,
    "recursive": broken_recursive,
    "by_k": broken_by_k,
    "reverse_bfs": broken_reverse_bfs,
    "bfs": broken_bfs,
    "Solution": lambda s, t: Solution().brokenCalc(s, t),
    "SolutionGreedy": lambda s, t: SolutionGreedy().brokenCalc(s, t),
    "SolutionRecursive": lambda s, t: SolutionRecursive().brokenCalc(s, t),
    "SolutionByK": lambda s, t: SolutionByK().brokenCalc(s, t),
    "SolutionReverseBFS": lambda s, t: SolutionReverseBFS().brokenCalc(s, t),
    "SolutionBFS": lambda s, t: SolutionBFS().brokenCalc(s, t),
}
SLOW = {"bfs", "SolutionBFS", "reverse_bfs", "SolutionReverseBFS"}


def check(s, t, want=None, names=None, slow_limit=3000):
    if want is None and max(s, t) <= slow_limit:
        want = broken_bfs(s, t)
    for name in (names or IMPLS):
        if name in SLOW and max(s, t) > slow_limit:
            continue
        got = IMPLS[name](s, t)
        if want is not None:
            assert got == want, (name, s, t, got, want)
    return want


for (s, t), want in examples:
    check(s, t, want)
print("ví dụ đề: BFS thuận và", len(IMPLS), "cài đặt đúng")

ví dụ đề: BFS thuận và 11 cài đặt đúng


### B. Edge case

- `startValue == target` (`0` phép).
- `target < startValue` (chỉ trừ), kể cả `target = 1`.
- `startValue = 1`: `target = 1`, `2`, `3`, … (luỹ thừa của 2: `log₂ t` phép).
- `target` là luỹ thừa của 2; `target` lẻ lớn.
- `target = startValue + 1`.
- Giá trị lớn nhất `10^9`.
- `(1, 10^9)`, `(10^9, 1)`, `(10^9, 10^9)`.

In [19]:
edge = [(1, 1), (2, 2), (5, 5), (10, 1), (100, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 7), (1, 8), (1, 1000), (3, 4), (4, 3), (2, 3), (5, 8), (3, 10),
        (7, 100), (1024, 1), (1, 1024), (9, 100), (999, 1000), (1000, 999), (6, 7), (6, 13)]
for s, t in edge:
    check(s, t)
big = [(1, 10 ** 9), (10 ** 9, 1), (10 ** 9, 10 ** 9), (2, 10 ** 9), (5, 10 ** 9 - 1), (10 ** 9 - 1, 10 ** 9), (3, 2 ** 29), (7, 2 ** 29 + 1)]
for s, t in big:
    results = {name: IMPLS[name](s, t) for name in ("greedy", "recursive", "by_k", "Solution", "SolutionGreedy", "SolutionRecursive", "SolutionByK")}
    assert len(set(results.values())) == 1, (s, t, results)
print(len(edge), "edge case nhỏ (BFS thuận kiểm) +", len(big), "edge case lớn (các cách O(log) khớp nhau): đúng")
print("f(1, 10^9) =", Solution().brokenCalc(1, 10 ** 9), "| f(3, 2^29) =", Solution().brokenCalc(3, 2 ** 29))

25 edge case nhỏ (BFS thuận kiểm) + 8 edge case lớn (các cách O(log) khớp nhau): đúng
f(1, 10^9) = 39 | f(3, 2^29) = 29


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_forward_greedy(s, t):                        # thuận: nhân đôi tới khi >= t rồi trừ
    ops = 0
    while s < t:
        s *= 2
        ops += 1
    return ops + s - t


def bug_odd_minus_one(s, t):                         # lẻ: trừ 1 thay vì cộng 1
    ops = 0
    while t > s:
        if t % 2:
            t -= 1
        else:
            t //= 2
        ops += 1
    return ops + s - t


def bug_no_final_diff(s, t):                         # quên cộng s - t ở cuối
    ops = 0
    while t > s:
        t = t + 1 if t % 2 else t // 2
        ops += 1
    return ops


def bug_infinite_loop(s, t):                         # while t != s: lặp vô hạn khi t < s (có giới hạn bước)
    ops, steps = 0, 0
    while t != s:
        steps += 1
        if steps > 200:
            raise RuntimeError("vòng lặp vô hạn")
        t = t + 1 if t % 2 else t // 2
        ops += 1
    return ops


def bug_floor_odd(s, t):                             # chia nguyên cả số lẻ (không +1)
    ops = 0
    while t > s:
        t //= 2
        ops += 1
    return ops + s - t


def bug_plus_one_even(s, t):                         # số chẵn cũng +1 (thay vì chia đôi) rồi mới chia: lãng phí phép
    ops, steps = 0, 0
    while t > s:
        steps += 1
        if steps > 500:
            raise RuntimeError("vòng lặp vô hạn")
        if t % 2 == 0:
            t += 1
            ops += 1
        else:
            t += 1
            ops += 1
        t //= 2
        ops += 1
    return ops + s - t


def bug_only_decrement(s, t):                        # chỉ dùng trừ 1 (bỏ nhân đôi)
    return s - t if s >= t else -1


BUGS = [bug_forward_greedy, bug_odd_minus_one, bug_no_final_diff, bug_infinite_loop, bug_floor_odd, bug_plus_one_even, bug_only_decrement]
pool = [(s, t) for s in range(1, 41) for t in range(1, 41)]
for bug in BUGS:
    found = None
    for s, t in pool:
        try:
            got = bug(s, t)
        except RuntimeError:
            got = "RuntimeError"
        if got != broken_bfs(s, t):
            found = (s, t, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:26s} sai trên (s, t) = ({found[0]}, {found[1]}): ra {found[2]}, đúng {broken_bfs(found[0], found[1])}")

bug_forward_greedy         sai trên (s, t) = (1, 5): ra 6, đúng 5
bug_odd_minus_one          sai trên (s, t) = (1, 3): ra 2, đúng 3
bug_no_final_diff          sai trên (s, t) = (2, 1): ra 0, đúng 1
bug_infinite_loop          sai trên (s, t) = (3, 1): ra RuntimeError, đúng 2
bug_floor_odd              sai trên (s, t) = (1, 3): ra 1, đúng 3
bug_plus_one_even          sai trên (s, t) = (1, 2): ra 2, đúng 1
bug_only_decrement         sai trên (s, t) = (1, 2): ra -1, đúng 1


### D. Vét cạn mọi cặp `1 ≤ s, t ≤ 300`

Với mỗi `s` ta chạy **một** BFS thuận từ `s` trên các giá trị `[1, U]` (`U = 1200 + 4 = 1204`... ta dùng `U = 4·300 + 4`) để có khoảng cách tới mọi `t ≤ 300`. Tổng `300 × 300 = 90 000` cặp. Ta cũng kiểm chứng rằng **nâng `U` gấp đôi** không đổi đáp án (oracle không bị chặn giới hạn ảnh hưởng).

In [21]:
def bfs_all(start, U):
    dist = {start: 0}
    queue = deque([start])
    while queue:
        v = queue.popleft()
        for w in (2 * v, v - 1):
            if 1 <= w <= U and w not in dist:
                dist[w] = dist[v] + 1
                queue.append(w)
    return dist


N = 300
t0 = time.perf_counter()
fast = ["greedy", "recursive", "by_k", "Solution", "SolutionGreedy", "SolutionRecursive", "SolutionByK"]
cnt = 0
max_f = 0
for s in range(1, N + 1):
    d1 = bfs_all(s, 4 * N + 4)
    d2 = bfs_all(s, 8 * N + 8)                                     # chặn lớn gấp đôi
    for t in range(1, N + 1):
        want = d1[t]
        assert d2[t] == want, (s, t)                               # chặn đủ lớn: đáp án ổn định
        for name in fast:
            assert IMPLS[name](s, t) == want, (name, s, t, IMPLS[name](s, t), want)
        cnt += 1
        max_f = max(max_f, want)
print(cnt, "cặp (s, t ≤ 300):", len(fast), "cài đặt khớp oracle BFS thuận; đáp án lớn nhất:", max_f, ";", round(time.perf_counter() - t0, 1), "s")
assert cnt == 90000

90000 cặp (s, t ≤ 300): 7 cài đặt khớp oracle BFS thuận; đáp án lớn nhất: 299 ; 5.9 s


In [22]:
# BFS ngược (có chặn) và các class BFS: kiểm trên miền nhỏ hơn
t0 = time.perf_counter()
cnt = 0
for s in range(1, 101):
    d = bfs_all(s, 4 * 100 + 4)
    for t in range(1, 101):
        for name in ("reverse_bfs", "SolutionReverseBFS", "SolutionBFS", "bfs"):
            assert IMPLS[name](s, t) == d[t], (name, s, t)
        cnt += 1
print(cnt, "cặp (s, t ≤ 100): BFS thuận, BFS ngược và các class BFS khớp;", round(time.perf_counter() - t0, 1), "s")

10000 cặp (s, t ≤ 100): BFS thuận, BFS ngược và các class BFS khớp; 1.2 s


### E. Tính chất / ngẫu nhiên (tới `10^9`, seed cố định)

(1) Các cách `O(log)` khớp nhau; (2) **dựng đường đi cụ thể (dạng nén các lượt trừ)** từ tham lam ngược rồi **phát lại**: kết thúc đúng `target` với đúng `f(s, t)` phép (số phép có thể tới `~5·10^8` nên không liệt kê từng phép); (3) **cận dưới** độc lập: công thức theo `k` (Cách 5) bằng tham lam; (4) `f(s, t) ≤ f(s, t + 1) + 1` (từ `t + 1` ta... thực chất `f(s, t) ≤ f(s, t+1) + 1` vì `t` đạt được từ `t + 1` bằng `−1`); (5) `f(s, 2t) ≤ f(s, t) + 1`; (6) `f(s, t) ≤ s − t` nếu `s ≥ t` và `f(s, t) ≤ 2·bit_length(t) + 2·...`; (7) `f(s, t) ≥ 1` nếu `s ≠ t`.

In [23]:
def construct_path(s, t):
    """Dựng dãy giá trị thuận s -> ... -> t từ tham lam ngược."""
    ops = []
    cur = t
    while cur > s:
        if cur % 2:
            cur += 1
            ops.append("-1")
        else:
            cur //= 2
            ops.append("*2")
    ops = ["-1"] * (s - cur) + ops[::-1]
    seq = [s]
    for op in ops:
        seq.append(seq[-1] * 2 if op == "*2" else seq[-1] - 1)
    return ops, seq


def construct_runs(s, t):
    """Đường đi thuận dạng NÉN: danh sách (phép, số lần); dùng cho giá trị lớn (số phép có thể ~5·10^8)."""
    tail, cur = [], t
    while cur > s:
        if cur % 2:
            cur += 1
            tail.append("-1")
        else:
            cur //= 2
            tail.append("*2")
    runs = [("-1", s - cur)] if s - cur else []
    for op in reversed(tail):
        if runs and runs[-1][0] == op == "-1":
            runs[-1] = ("-1", runs[-1][1] + 1)
        else:
            runs.append((op, 1))
    return runs


def replay_runs(s, runs):
    v, total = s, 0
    for op, c in runs:
        if op == "-1":
            v -= c
        else:
            v *= 2
        assert v >= 1, "giá trị trung gian không dương"
        total += c
    return v, total


rng = random.Random(20240991)
t0 = time.perf_counter()
for it in range(4000):
    s = rng.choice([rng.randint(1, 10 ** 9), rng.randint(1, 30), rng.randint(1, 1000)])
    t = rng.choice([rng.randint(1, 10 ** 9), rng.randint(1, 100), 2 ** rng.randint(0, 29), 2 ** rng.randint(0, 29) + rng.choice([-1, 1])])
    t = max(1, t)
    r = {name: IMPLS[name](s, t) for name in fast}
    assert len(set(r.values())) == 1, (s, t, r)                                       # (1) (3)
    f = r["Solution"]
    v_end, total = replay_runs(s, construct_runs(s, t))                               # (2) phát lại đường đi nén
    assert v_end == t and total == f, (s, t, v_end, total, f)
    assert f <= broken_greedy(s, t + 1) + 1                                           # (4)
    assert broken_greedy(s, 2 * t) <= f + 1 if 2 * t <= 2 * 10 ** 9 else True          # (5)
    if s >= t:
        assert f == s - t                                                             # (6)
    assert (f >= 1) == (s != t)                                                       # (7)
print("4000 cặp ngẫu nhiên tới 10^9: 7 tính chất đúng; đường đi dựng được phát lại đúng;", round(time.perf_counter() - t0, 1), "s")

4000 cặp ngẫu nhiên tới 10^9: 7 tính chất đúng; đường đi dựng được phát lại đúng; 0.3 s


In [24]:
# cặp nhỏ ngẫu nhiên (tới 2000): so với oracle BFS thuận cho mọi cài đặt (kể cả BFS ngược)
rng = random.Random(99102)
t0 = time.perf_counter()
for _ in range(1200):
    s, t = rng.randint(1, 2000), rng.randint(1, 2000)
    want = broken_bfs(s, t)
    for name in IMPLS:
        assert IMPLS[name](s, t) == want, (name, s, t)
print("1200 cặp ngẫu nhiên ≤ 2000: mọi cài đặt (kể cả BFS thuận/ngược) khớp;", round(time.perf_counter() - t0, 1), "s")

1200 cặp ngẫu nhiên ≤ 2000: mọi cài đặt (kể cả BFS thuận/ngược) khớp; 3.7 s


### F. Giới hạn

`startValue`, `target` tới `10^9`: `(1, 10^9)`, `(10^9, 1)`, `(10^9, 10^9)`, luỹ thừa của 2 và `±1`, mọi `(1, t)` với `t = 2^k ± 1`; số bước tối đa quanh `2·log₂(10^9) ≈ 60`.

In [25]:
t0 = time.perf_counter()
fastf = ["greedy", "recursive", "by_k", "Solution", "SolutionGreedy", "SolutionRecursive", "SolutionByK"]
cases = []
for k in range(0, 30):
    for dt in (-1, 0, 1):
        t = 2 ** k + dt
        if 1 <= t <= 10 ** 9:
            cases += [(1, t), (3, t), (10 ** 9, t), (t, 10 ** 9)]
max_ops = 0
for s, t in cases:
    r = {name: IMPLS[name](s, t) for name in fastf}
    assert len(set(r.values())) == 1, (s, t, r)
    max_ops = max(max_ops, r["Solution"])
    v_end, total = replay_runs(s, construct_runs(s, t))
    assert v_end == t and total == r["Solution"], (s, t)
print(len(cases), "cặp quanh luỹ thừa của 2 và 10^9: các cách O(log) khớp; số phép lớn nhất khi s <= t:", max_ops, ";", round(time.perf_counter() - t0, 2), "s")
assert Solution().brokenCalc(1, 10 ** 9) == broken_by_k(1, 10 ** 9)

356 cặp quanh luỹ thừa của 2 và 10^9: các cách O(log) khớp; số phép lớn nhất khi s <= t: 999999999 ; 0.03 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: tham lam thuận (nhân đôi tới khi vượt rồi trừ)

In [26]:
print("bug:", bug_forward_greedy(5, 8), "| đúng:", Solution().brokenCalc(5, 8))
assert bug_forward_greedy(5, 8) == 3 and Solution().brokenCalc(5, 8) == 2

bug: 3 | đúng: 2


**Vì sao sai:** từ `5`, nhân đôi tới `10` rồi trừ 2 lần = 3 phép; nhưng trừ **trước** (`5 → 4`) rồi nhân đôi (`→ 8`) chỉ 2 phép. Thứ tự các phép quan trọng; tham lam thuận không thấy được. **Đúng:** đi **ngược** từ `target`.

### Lỗi 2: số lẻ ngược lại `−1` thay vì `+1`

In [27]:
print("bug:", bug_odd_minus_one(1, 3), "| đúng:", Solution().brokenCalc(1, 3))
assert bug_odd_minus_one(1, 3) == 2 and Solution().brokenCalc(1, 3) == 3 == broken_bfs(1, 3)   # 1 -> 2 -> 4 -> 3

bug: 2 | đúng: 3


**Vì sao sai:** phép ngược của "trừ 1" là **cộng 1** (không phải trừ 1): số lẻ `t` do `t + 1` trừ 1 mà ra.

### Lỗi 3: quên cộng `startValue − target` ở cuối

In [28]:
print("bug:", bug_no_final_diff(1024, 1), "| đúng:", Solution().brokenCalc(1024, 1))
assert bug_no_final_diff(1024, 1) == 0 and Solution().brokenCalc(1024, 1) == 1023

bug: 0 | đúng: 1023


**Vì sao sai:** khi `target ≤ startValue` vẫn cần `startValue − target` phép trừ.

### Lỗi 4: `while target != startValue` (vòng lặp vô hạn)

In [29]:
try:
    bug_infinite_loop(10, 3)
    print("không lỗi?")
except RuntimeError as e:
    print("bug:", e)
print("đúng:", Solution().brokenCalc(10, 3))

bug: vòng lặp vô hạn
đúng: 7


**Vì sao sai:** nếu `target < startValue` thì `target` có thể nhảy qua `startValue` (hoặc không bao giờ bằng). **Đúng:** điều kiện `while target > startValue`.

### Lỗi 5: chia nguyên số lẻ

In [30]:
print("bug:", bug_floor_odd(1, 3), "| đúng:", Solution().brokenCalc(1, 3))
assert bug_floor_odd(1, 3) == 1 and Solution().brokenCalc(1, 3) == 3 == broken_bfs(1, 3)      # 1 -> 2 -> 4 -> 3

bug: 1 | đúng: 3


**Vì sao sai:** `11 // 2 = 5` không phải phép ngược của bất kỳ phép nào (nhân đôi `5` ra `10`). **Đúng:** số lẻ phải `+1` rồi chia.

### Lỗi 6 (bonus): chỉ dùng phép trừ

In [31]:
print("bug:", bug_only_decrement(5, 8), "| đúng:", Solution().brokenCalc(5, 8))
assert bug_only_decrement(5, 8) == -1

bug: -1 | đúng: 2


**Vì sao sai:** muốn tăng giá trị bắt buộc dùng nhân đôi.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "biến đổi bằng vài phép (×2, −1) với số phép tối thiểu" → **nghĩ ngược** từ đích; phép ngược của ×2 là ÷2 (chỉ khi chẵn) và của −1 là +1.

**Mẹo:**

1. Số lẻ ⇒ `+1` bắt buộc; số chẵn ⇒ chia đôi.
2. Điều kiện dừng `target ≤ startValue`, rồi cộng hiệu.
3. Tham lam thuận thường sai khi phép giảm có thể đặt **trước** phép nhân.
4. Oracle: BFS thuận có chặn giá trị trên miền nhỏ.
5. Cách nhìn đổi tiền: số lần nhân đôi `k` cố định ⇒ tối thiểu hoá số lần trừ bằng mệnh giá `2^i`.
6. Dựng đường đi cụ thể để kiểm chứng.

**Họ bài cùng pattern:** Minimum Operations to Reduce X to Zero (1658), Minimum Number of Steps to Make Two Strings Anagram, 2 Keys Keyboard (650), Integer Replacement (397), Minimum Moves to Reach Target Score (2139), Minimum Operations to Convert Number (2059).

## 15. Biến thể

1. **Trả về dãy phép** (đường đi cụ thể) thay vì chỉ số phép.
2. **Nhân `m` thay vì nhân đôi** (`×m` hoặc `−1`): tổng quát hoá (cần kiểm chứng bằng BFS).
3. **Cho phép cộng 1 thay vì trừ 1** (`×2` hoặc `+1`) từ `start ≤ target`: đảo vai trò.

Cài đặt và đối chiếu BFS:

In [32]:
def broken_ops(s, t):
    ops, seq = construct_path(s, t)
    return ops


def broken_m(start, target, m):
    """Biến thể 2: phép ×m hoặc −1; tham lam ngược: nếu target % m != 0 thì +1 (mỗi lần 1 phép), nếu chia hết thì chia m."""
    ops = 0
    while target > start:
        if target % m:
            ops += 1
            target += 1
        else:
            ops += 1
            target //= m
    return ops + start - target


def bfs_m(start, target, m):
    U = 4 * max(start, target) + 4
    dist = {start: 0}
    queue = deque([start])
    while queue:
        v = queue.popleft()
        if v == target:
            return dist[v]
        for w in (m * v, v - 1):
            if 1 <= w <= U and w not in dist:
                dist[w] = dist[v] + 1
                queue.append(w)
    return -1


def plus_double(start, target):
    """Biến thể 3: phép ×2 hoặc +1 (start <= target): ngược lại ÷2 (chẵn) hoặc −1."""
    ops = 0
    while target > start:
        if target % 2 == 0 and target // 2 >= start:
            target //= 2
        else:
            target -= 1
        ops += 1
    return ops if target == start else -1


def bfs_plus_double(start, target):
    dist = {start: 0}
    queue = deque([start])
    while queue:
        v = queue.popleft()
        if v == target:
            return dist[v]
        for w in (2 * v, v + 1):
            if w <= target and w not in dist:
                dist[w] = dist[v] + 1
                queue.append(w)
    return -1


for s in range(1, 60):
    for t in range(1, 80):
        ops, seq = construct_path(s, t)
        assert len(ops) == broken_bfs(s, t) and seq[-1] == t                    # (1)
        assert plus_double(s, t) == bfs_plus_double(s, t) if s <= t else True    # (3)
mismatch = [(s, t, m) for m in (3, 4, 5) for s in range(1, 25) for t in range(1, 60) if broken_m(s, t, m) != bfs_m(s, t, m)]
print("đường đi cụ thể và ×2/+1: khớp BFS; tổng quát ×m (m = 3, 4, 5): số cặp lệch BFS =", len(mismatch), (mismatch[:3] if mismatch else ""))

đường đi cụ thể và ×2/+1: khớp BFS; tổng quát ×m (m = 3, 4, 5): số cặp lệch BFS = 0 


**Nhận xét (biến thể 2):** trong miền đã thử (`m = 3, 4, 5`, `startValue < 25`, `target < 60`) tham lam ngược **không** lệch BFS lần nào (số cặp lệch in ra ở trên là `0`). Đây chỉ là bằng chứng thực nghiệm trên miền nhỏ, không phải chứng minh; ở `m ≥ 3` khi `target` không chia hết cho `m` ta phải `+1` nhiều lần (tối đa `m − 1`) và lập luận (c) cần được tổng quát hoá cẩn thận.

In [33]:
assert all(broken_m(s, t, 2) == broken_bfs(s, t) for s in range(1, 40) for t in range(1, 80))
print("m = 2: tham lam ngược khớp BFS trên 3 081 cặp")

m = 2: tham lam ngược khớp BFS trên 3 081 cặp


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS thuận | khoảng cách ngắn nhất | `O(U)` | `O(U)` | oracle giá trị nhỏ |
| 2. Tham lam ngược (lặp) | lẻ `+1`, chẵn `÷2` | `O(log t)` | `O(1)` | **đáp án chính** |
| 3. Tham lam ngược (đệ quy) | cùng ý | `O(log t)` | `O(log t)` | gọn |
| 4. BFS ngược có chặn | hai phép ngược | `O(max)` | `O(max)` | kiểm chứng không cần tham lam |
| 5. Công thức theo `k` | đổi tiền luỹ thừa 2 | `O(log² t)` | `O(1)` | cách nhìn độc lập |

Mọi cách khớp oracle BFS thuận trên `90 000` cặp; các cách `O(log)` khớp nhau tới `10^9`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [34]:
print(Solution().brokenCalc(2, 3), Solution().brokenCalc(5, 8), Solution().brokenCalc(3, 10), Solution().brokenCalc(1024, 1), Solution().brokenCalc(1, 16))

2 2 3 1023 4


### Bài 2 (tính tay)

`startValue = 4`, `target = 13`. (a) Đi ngược từ `13` về `4`, ghi từng phép. (b) Tổng số phép? (c) Viết đường đi thuận. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện tham lam ngược:

In [35]:
def greedy_todo(start, target):
    ops = 0
    while target > start:
        pass            # TODO 1: nếu target lẻ thì target += 1, ngược lại target //= 2
        pass            # TODO 2: ops += 1
        break           # xoá dòng này khi làm xong
    return 0            # TODO 3: return ops + start - target


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(greedy_todo(s, t) == w for (s, t), w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **công thức theo số lần nhân đôi `k`**:

In [36]:
class MySolution:
    def brokenCalc(self, startValue, target):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().brokenCalc(s, t) == w for (s, t), w in examples)

### Bài 5 (mini challenge): dãy phép cụ thể

Viết `sequence(start, target)` trả về **danh sách các giá trị** hiển thị trên máy theo một dãy tối ưu (bắt đầu `start`, kết thúc `target`). Kiểm bằng cách phát lại và so độ dài với `brokenCalc`.

In [37]:
def sequence(start, target):
    pass            # TODO


# Khi làm xong:
# seq = sequence(3, 10)
# assert seq[0] == 3 and seq[-1] == 10 and len(seq) - 1 == 3

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đi ngược từ `target`.
- Mức 2: `(1, 16)`: `16 → 8 → 4 → 2 → 1`: 4 phép chia đôi.
- Mức 3: `2 2 3 1023 4`.

**Bài 2**

- Mức 1: `13` lẻ ⇒ `+1`.
- Mức 2: `13 → 14 → 7 → 8 → 4`: dừng khi `target ≤ 4`.
- Mức 3: 4 phép; đường đi thuận `4 → 8 → 7 → 14 → 13`.

**Bài 3**

- Mức 1: `if target % 2: target += 1 else: target //= 2`.
- Mức 2: mỗi lần lặp `ops += 1`.
- Mức 3: `return ops + start - target`.

**Bài 4**

- Mức 1: thử `k = 0..40`.
- Mức 2: `D = start * 2**k - target`; chỉ xét `D >= 0`.
- Mức 3: `cost = k + D // 2**k + bin(D % 2**k).count("1")`; lấy `min`.

**Bài 5**

- Mức 1: dựng danh sách phép ngược, rồi đảo.
- Mức 2: thêm `start - target` phép trừ ở đầu nếu `target < start` sau vòng lặp.
- Mức 3: phát lại: `*2` hoặc `-1` trên dãy giá trị.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [38]:
vals = (Solution().brokenCalc(2, 3), Solution().brokenCalc(5, 8), Solution().brokenCalc(3, 10), Solution().brokenCalc(1024, 1), Solution().brokenCalc(1, 16))
print(vals)
assert vals == (2, 2, 3, 1023, 4)
print("Bài 1 OK")

(2, 2, 3, 1023, 4)
Bài 1 OK


### Bài 2

In [39]:
s, t = 4, 13
chain = [t]
cur = t
while cur > s:
    cur = cur + 1 if cur % 2 else cur // 2
    chain.append(cur)
print("đường ngược:", chain, "| số phép:", len(chain) - 1 + (s - cur))
assert chain == [13, 14, 7, 8, 4] and Solution().brokenCalc(s, t) == 4 == broken_bfs(s, t)
ops, seq = construct_path(s, t)
assert seq == [4, 8, 7, 14, 13]
print("Bài 2 OK: đường đi thuận", seq)

đường ngược: [13, 14, 7, 8, 4] | số phép: 4
Bài 2 OK: đường đi thuận [4, 8, 7, 14, 13]


### Bài 3

In [40]:
def greedy_answer(start, target):
    ops = 0
    while target > start:
        if target % 2:
            target += 1
        else:
            target //= 2
        ops += 1
    return ops + start - target


assert all(greedy_answer(s, t) == w for (s, t), w in examples)
for s in range(1, 60):
    for t in range(1, 60):
        assert greedy_answer(s, t) == broken_bfs(s, t)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [41]:
class MySolutionAnswer:
    def brokenCalc(self, startValue, target):
        best = None
        for k in range(0, 40):
            D = startValue * (1 << k) - target
            if D >= 0:
                c = k + D // (1 << k) + bin(D % (1 << k)).count("1")
                best = c if best is None else min(best, c)
        return best


assert all(MySolutionAnswer().brokenCalc(s, t) == w for (s, t), w in examples)
for s in range(1, 80):
    for t in range(1, 80):
        assert MySolutionAnswer().brokenCalc(s, t) == broken_bfs(s, t)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [42]:
def sequence_answer(start, target):
    ops, cur = [], target
    while cur > start:
        if cur % 2:
            cur += 1
            ops.append("-1")
        else:
            cur //= 2
            ops.append("*2")
    ops = ["-1"] * (start - cur) + ops[::-1]
    seq = [start]
    for op in ops:
        seq.append(seq[-1] * 2 if op == "*2" else seq[-1] - 1)
    return seq


seq = sequence_answer(3, 10)
assert seq == [3, 6, 5, 10] and len(seq) - 1 == Solution().brokenCalc(3, 10)
for s in range(1, 50):
    for t in range(1, 50):
        q = sequence_answer(s, t)
        assert q[0] == s and q[-1] == t and len(q) - 1 == Solution().brokenCalc(s, t)
print("Bài 5 OK:", seq)

Bài 5 OK: [3, 6, 5, 10]


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (biến đổi bằng ×2 và −1 với số phép tối thiểu → đi ngược từ đích):**

```python
class Solution:
    def brokenCalc(self, startValue: int, target: int) -> int:
        ops = 0
        while target > startValue:
            target = target + 1 if target % 2 else target // 2
            ops += 1
        return ops + startValue - target
```

**Checklist trước khi submit:**

- [ ] Đi **ngược** từ `target` (không tham lam thuận)?
- [ ] Lẻ ⇒ `+1`; chẵn ⇒ `//2`?
- [ ] Điều kiện `while target > startValue` (không `!=`)?
- [ ] Cộng `startValue − target` ở cuối?
- [ ] Đã thử `(2,3)=2`, `(5,8)=2`, `(3,10)=3`, `(1024,1)=1023`, `(1, 10^9)`?
- [ ] Hiểu vì sao chia đôi luôn tốt hơn `+1` khi `target` chẵn?
- [ ] Đúng tên `class Solution`, `brokenCalc(self, startValue, target)`, không in debug.